In [1]:
import torch
import torchxrayvision as xrv
import torch.nn as nn
import numpy as np

In [2]:
KFOLDS = 5
THRESHOLD = 0.5
NUM_EPOCHS = 10
SUBSET_FRAC = None
BATCH_SIZE = 32
NUM_WORKERS = 8

In [3]:
from dataloading.load_nih import load_nih

nih = load_nih()

Using metadata: data/nih-chest-xrays/Data_Entry_2017.csv
Matched rows with images: 112,120


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def build_model():
    model = xrv.models.DenseNet(weights='densenet121-res224-all', drop_rate=0.2).to(device)

    # Freeze all early layers for transfer learning
    for param in model.parameters():
        param.requires_grad = False

    # Optional: Unfreeze the last blocks for fine-tuning
    # For DenseNet, this is 'features.denseblock4'
    # for param in model.features.denseblock4.parameters():
    #     param.requires_grad = True

    # Replace the classification head (DenseNet uses 'classifier', not 'fc')
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, nih.num_classes)
    model.op_threshs = None
    model = model.to(device)

    return model


Using device: cuda


In [5]:
# Cell 8: Train/validation loop utilities
from tqdm.auto import tqdm


def binary_metrics(logits: torch.Tensor, targets: torch.Tensor, threshold: float = 0.5, eps: float = 1e-8):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    tp = (preds * targets).sum()
    fp = (preds * (1 - targets)).sum()
    fn = ((1 - preds) * targets).sum()
    tn = ((1 - preds) * (1 - targets)).sum()

    accuracy = (tp + tn) / (tp + tn + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = (2 * precision * recall) / (precision + recall + eps)
    return accuracy.item(), f1.item()

def run_epoch(model, optimizer, criterion, loader, train_mode=True):
    model.train() if train_mode else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_f1 = 0.0
    steps = 0

    pbar = tqdm(loader, desc="Train" if train_mode else "Val", leave=False)
    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, targets)

                if train_mode:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    optimizer.step()

        acc, f1 = binary_metrics(logits.detach(), targets, threshold=THRESHOLD)
        
        total_loss += loss.item()
        total_acc += acc
        total_f1 += f1
        steps += 1
        
        pbar.set_postfix(loss=f"{loss.item():.4f}", f1=f"{f1:.4f}")

    return (
        total_loss / max(steps, 1),
        total_acc / max(steps, 1),
        total_f1 / max(steps, 1),
    )

In [6]:
from sklearn.model_selection import KFold, train_test_split
from dataloading.load_nih import NIHChestDataset, get_transforms
from torch.utils.data import DataLoader

# 5-Fold Cross Validation Setup
kf = KFold(n_splits=KFOLDS, shuffle=True, random_state=42)
train_tfms, val_tfms = get_transforms()

fold_histories = []
best_fold_f1s = []

print(f"Starting {KFOLDS}-Fold Cross Validation...")

X_cv, X_test = train_test_split(nih.id_df, test_size=0.2, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_cv)):
    print(f"\n{'='*20} FOLD {fold + 1}/{KFOLDS} {'='*20}")
    
    # 1. Split Data
    fold_train_df = X_cv.iloc[train_idx]
    fold_val_df = X_cv.iloc[val_idx]
    
    # 2. Setup DataLoaders
    train_ds = NIHChestDataset(fold_train_df, transform=train_tfms)
    val_ds = NIHChestDataset(fold_val_df, transform=val_tfms)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=torch.cuda.is_available(), num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=torch.cuda.is_available(), num_workers=NUM_WORKERS)
    
    # 3. Initialize Fresh Model
    model = build_model()
    
    # 4. Handle Imbalance for Current Fold
    train_targets = np.stack(fold_train_df["target"].values).astype(np.float32)
    pos_count = train_targets.sum(axis=0)
    neg_count = len(train_targets) - pos_count
    pos_weight_value = torch.tensor(np.divide(neg_count, np.maximum(pos_count, 1.0)), dtype=torch.float32, device=device)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_value)
    
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-2)
    
    # 5. Train Model
    best_val_f1 = -1.0
    history = []
    
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc, train_f1 = run_epoch(model, optimizer, criterion, train_loader, train_mode=True)
        val_loss, val_acc, val_f1 = run_epoch(model, optimizer, criterion, val_loader, train_mode=False)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
        })

        print(
            f"Fold {fold+1} Epoch [{epoch}/{NUM_EPOCHS}] "
            f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
            f"val_loss={val_loss:.4f} val_f1={val_f1:.4f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), f"best_densenet50_fold_{fold+1}.pth")
            print(f"  -> Saved best model for Fold {fold+1} (val_f1={best_val_f1:.4f})")
            
    fold_histories.append(history)
    best_fold_f1s.append(best_val_f1)
    print(f"Fold {fold+1} finished. Best F1: {best_val_f1:.4f}")

print("\n" + "="*30)
print(f"Cross Validation Complete! Mean Best F1: {np.mean(best_fold_f1s):.4f}")

Starting 5-Fold Cross Validation...

==================== FOLD 1/5 ====================


Train:   0%|          | 0/1036 [00:23<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import pickle

with open("densenet121-training-histories.pkl", "wb") as fp:
    pickle.dump(fold_histories, fp)

In [ ]:
def calibrate(model):
    model.eval()

    for module in model.modules():
        if module.__class__.__name__ == '_DenseLayer':
            module.training = True

In [ ]:
from tqdm import tqdm

def entropy(p):
    # Add epsilon to prevent log(0). 
    # Computes per-sample entropy across classes. Shape: (batch_size,)
    eps = 1e-8
    return -torch.sum(p * torch.log(p + eps), dim=-1)

def epistemic_uncertainty(preds):
    # preds shape: (N_models_or_passes, batch_size, num_classes)
    
    # 1. Total entropy: entropy of the mean prediction
    p_mean = torch.mean(preds, dim=0) # (batch_size, num_classes)
    tot_ent = entropy(p_mean)         # (batch_size,)
    
    # 2. Aleatoric uncertainty: mean of the entropies
    aleatoric = torch.mean(entropy(preds), dim=0) # (batch_size,)
    
    # 3. Epistemic uncertainty
    return tot_ent - aleatoric

def compute_id_scores(id_loader, num_passes=10):
    # Pre-load models ONCE outside the batch loop to prevent massive disk I/O bottlenecks!
    models = []
    for fold in range(KFOLDS):
        m = build_model()
        m.load_state_dict(torch.load(f"best_densenet50_fold_{fold+1}.pth", map_location=device))
        calibrate(m)
        models.append(m)

    id_scores = []
    with torch.no_grad():
        for images, _ in tqdm(id_loader, "Computing ID scores"):
            images = images.to(device)
            
            outputs = []
            for m in models:
                # MCDropout: Pass data multiple times through the SAME model
                for _ in range(num_passes):
                    logits = m(images)
                    probs = torch.sigmoid(logits) # Convert logits to probabilities!
                    outputs.append(probs)
            
            # Stack all multi-model outputs 
            # New shape: (KFOLDS * num_passes, batch_size, num_classes)
            preds = torch.stack(outputs)
            
            # Vectorized epistemic uncertainty -> shape: (batch_size,)
            uncertainty = epistemic_uncertainty(preds)
            
            # Negative uncertainty is used so higher score = more ID (less uncertain)
            id_scores.extend(-uncertainty.cpu().numpy())
            
    return id_scores

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_loader = DataLoader(
    NIHChestDataset(X_test, transform=val_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    num_workers=NUM_WORKERS
)

id_scores = compute_id_scores(test_loader)

Computing ID scores: 100%|██████████| 81/81 [10:53<00:00,  8.06s/it]


In [ ]:
id_scores

[np.float32(-0.0009851456),
 np.float32(-0.0044059753),
 np.float32(-0.0008883476),
 np.float32(-0.00106287),
 np.float32(-0.002471447),
 np.float32(-0.00071525574),
 np.float32(-0.0020303726),
 np.float32(-0.0016412735),
 np.float32(-0.0011520386),
 np.float32(-0.001557827),
 np.float32(-0.0009999275),
 np.float32(-0.0010514259),
 np.float32(-0.0025525093),
 np.float32(-0.0010757446),
 np.float32(-0.0014977455),
 np.float32(-0.0009737015),
 np.float32(-0.0017943382),
 np.float32(-0.00096178055),
 np.float32(-0.0008058548),
 np.float32(-0.0011725426),
 np.float32(-0.0008678436),
 np.float32(-0.004067898),
 np.float32(-0.0015563965),
 np.float32(-0.0053629875),
 np.float32(-0.0016317368),
 np.float32(-0.0010242462),
 np.float32(-0.0046420097),
 np.float32(-0.004123211),
 np.float32(-0.0008234978),
 np.float32(-0.0017232895),
 np.float32(-0.002175808),
 np.float32(-0.001522541),
 np.float32(-0.0013628006),
 np.float32(-0.0031166077),
 np.float32(-0.0008111),
 np.float32(-0.0021219254),
 

In [ ]:
def determine_threshold(id_scores, percentile=5):
    threshold = np.percentile(id_scores, percentile)
    return threshold

threshold = determine_threshold(id_scores, percentile=5)
threshold

np.float32(-0.016758919)

In [ ]:
df_ood_subset = nih.ood_df.sample(n = len(X_test) // KFOLDS, random_state=42)
ood_dataset = NIHChestDataset(df_ood_subset, transform=val_tfms)
ood_loader = DataLoader(
    ood_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    num_workers=NUM_WORKERS
)

In [ ]:
ood_scores = compute_id_scores(ood_loader)

Computing ID scores: 100%|██████████| 41/41 [05:27<00:00,  8.00s/it]


In [ ]:
from metrics import fpr
from sklearn.metrics import average_precision_score

y_true = [1] * len(id_scores) + [0] * len(ood_scores)
y_scores = id_scores + ood_scores

auprc = average_precision_score(y_true, y_scores)
fpr90 = fpr(y_true, y_scores, percentile=10)
fpr95 = fpr(y_true, y_scores, percentile=5)
fpr99 = fpr(y_true, y_scores, percentile=1)

print(f"AUPRC: {auprc:.4f}")
print(f"FPR at 90% TPR: {fpr90:.4f}")
print(f"FPR at 95% TPR: {fpr95:.4f}")
print(f"FPR at 99% TPR: {fpr99:.4f}")

AUPRC: 0.8075
FPR at 90% TPR: 0.7349
FPR at 95% TPR: 0.8640
FPR at 99% TPR: 0.9675


In [ ]:
from torchvision import datasets, transforms
import torch
from torch.utils.data import DataLoader, Subset

# Ensure all Caltech images are forced to RGB and use identical validation transforms
caltech_tfms = transforms.Compose([
    transforms.Lambda(lambda x: x.convert("RGB")),
    val_tfms
])

caltech_data = datasets.Caltech256(
    root='./data', 
    download=True, 
    transform=caltech_tfms
)

# Randomly sample the Caltech dataset to match the size of your Validation (ID) dataset
num_samples = min(len(X_test), len(caltech_data))
indices = torch.randperm(len(caltech_data))[:num_samples]
caltech_subset = Subset(caltech_data, indices.tolist())

caltech_loader = DataLoader(
    caltech_subset,
    batch_size=16,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    num_workers=NUM_WORKERS
)

print(f"Loaded {len(caltech_subset)} Caltech256 images to use as OOD.")

Loaded 5176 Caltech256 images to use as OOD.


In [ ]:
ood_scores_caltech = compute_id_scores(caltech_loader)

# Assign class labels exactly as in the previous cell
y_true_caltech = [1] * len(id_scores) + [0] * len(ood_scores_caltech)
y_scores_caltech = id_scores + ood_scores_caltech

auprc_caltech = average_precision_score(y_true_caltech, y_scores_caltech)
fpr90_caltech = fpr(y_true_caltech, y_scores_caltech, percentile=10)
fpr95_caltech = fpr(y_true_caltech, y_scores_caltech, percentile=5)
fpr99_caltech = fpr(y_true_caltech, y_scores_caltech, percentile=1)

print("=== ID (NIH X-Ray) vs OOD (Caltech256) ===")
print(f"AUPRC: {auprc_caltech:.4f}")
print(f"FPR at 90% TPR: {fpr90_caltech:.4f}")
print(f"FPR at 95% TPR: {fpr95_caltech:.4f}")
print(f"FPR at 99% TPR: {fpr99_caltech:.4f}")

Computing ID scores: 100%|██████████| 324/324 [12:10<00:00,  2.25s/it]

=== ID (NIH X-Ray) vs OOD (Caltech256) ===
AUPRC: 0.5147
FPR at 90% TPR: 0.8271
FPR at 95% TPR: 0.8688
FPR at 99% TPR: 0.9075


In [ ]:
ood_scores_caltech

[np.float32(-0.015890598),
 np.float32(-0.0054512024),
 np.float32(-0.007381439),
 np.float32(-0.013905048),
 np.float32(-0.0062761307),
 np.float32(-0.0075101852),
 np.float32(-0.009744644),
 np.float32(-0.009439945),
 np.float32(-0.0028104782),
 np.float32(-0.014815807),
 np.float32(-0.0031218529),
 np.float32(-0.014636517),
 np.float32(-0.010910511),
 np.float32(-0.008180141),
 np.float32(-0.0040955544),
 np.float32(-0.016447544),
 np.float32(-0.0025367737),
 np.float32(-0.007449627),
 np.float32(-0.0057964325),
 np.float32(-0.0030078888),
 np.float32(-0.0046567917),
 np.float32(-0.0044288635),
 np.float32(-0.008577347),
 np.float32(-0.004087925),
 np.float32(-0.021622181),
 np.float32(-0.008120537),
 np.float32(-0.0028896332),
 np.float32(-0.006640911),
 np.float32(-0.008772373),
 np.float32(-0.009563446),
 np.float32(-0.0124435425),
 np.float32(-0.0024704933),
 np.float32(-0.008913517),
 np.float32(-0.010192394),
 np.float32(-0.008187771),
 np.float32(-0.004529476),
 np.float32(-0